# 1. Data Collection 

*AI assistance (Claude-sonnet-4.6 in UvA AI Chat): notebook structure, test mode setup, and debugging the MediaWiki API `userid` field issue.*

This notebook collects the **Wikipedia data** needed for the project. It runs four steps in order:

| Step | Data | Output |
|------|------|--------|
| 1 | Newcomer accounts — `Special:Log/newusers` | `registration_log.jsonl` |
| 2 | Edit histories per newcomer (≥1 edit filter + session edits) | `newcomer_edits.jsonl` |
| 3 | Stratified random sample (`TARGET_SAMPLE_SIZE`) | `sample_index.json` |
| 4 | Talk-page revision histories per sampled newcomer | `talkpage_messages.jsonl` |

The **external dataset** (Zheng et al., 2019 bot-role classification file) is downloaded from [this link](https://github.com/adashiyj/Wikipedia_bots_taxonomy/blob/master/botrole_predict_and_proportion_of_edits.xlsx) and placed in the `Final project/data/raw/` folder.

**Design principles:**
- One shared `requests.Session` with `User-Agent` set once.
- All API calls go through `api_get()` with `maxlag` and back-off.
- Every API response is saved to disk immediately (JSONL append pattern).
- Re-running the notebook skips already-collected items without re-hitting the API.
- A fixed random seed (42) ensures reproducible sampling.


### 1.0 Configuration

In [1]:
import json
import os
import time
import random
from pathlib import Path
from datetime import datetime, timedelta
import requests
import pandas as pd

In [2]:
# ── Working directory ─────────────────────────────────────────────────────────
os.chdir("/Users/ada/bdacaII-2026-ada")

# ── Test mode ────────────────────────────────────────────────────────────────
# Set to True to run a quick sanity check on a tiny slice of data.
# Set to False for the real collection run.
TEST_MODE = False

# ── API endpoint ──────────────────────────────────────────────────────────────
API = "https://en.wikipedia.org/w/api.php"

# ── Sampling parameters ───────────────────────────────────────────────────────
TARGET_SAMPLE_SIZE = 10_000     # newcomers to analyse
MAX_LOG_ACCOUNTS   = 30_000     # stop fetching the log after this many accounts
RANDOM_SEED        = 42         # fixed for reproducibility

# ── Collection window: March–May 2025 (inclusive), following Zheng et al. (2019)
LOG_START = "2025-03-01T00:00:00Z"
LOG_END   = "2025-05-31T23:59:59Z"

# ── Retention and session windows ────────────────────────────────────────────
RETENTION_DAYS     = 60         # days after first edit to check retention
SESSION_WINDOW_MIN = 60         # minutes defining the first edit session

# ── Polite API call settings (Wikimedia bot policy) ──────────────────────────
# MAXLAG: cooperative DB-lag threshold in seconds. The API returns HTTP 503
# when the replica DB is behind by more than this value, triggering back-off.
# 5 s is the recommended value for non-urgent automated scripts.
MAXLAG              = 5     # seconds; cooperative DB-lag threshold
REQUEST_TIMEOUT     = 15    # seconds before a request is considered hung
RETRY_DELAY         = 2     # initial back-off delay in seconds (doubles each retry)
MAX_RETRY_DELAY     = 60    # back-off is capped at this value to avoid indefinite stalls
MAX_RETRIES         = 5     # maximum retry attempts per API call
INTER_REQUEST_DELAY = 0.1   # polite pause between successive API calls (seconds)
INTER_PAGE_DELAY    = 0.5   # polite pause between paginated API calls (seconds)

# ── Test mode overrides ───────────────────────────────────────────────────────
if TEST_MODE:
    TARGET_SAMPLE_SIZE = 10
    MAX_LOG_ACCOUNTS   = 50
    LOG_START = "2024-10-01T00:00:00Z"
    LOG_END   = "2024-12-31T23:59:59Z"

# ── Data directories ──────────────────────────────────────────────────────────
DATA_DIR      = Path("Final_project/data")
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CACHE_DIR     = RAW_DIR / "cache"

for directory in (RAW_DIR, PROCESSED_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# ── JSONL output paths ────────────────────────────────────────────────────────
LOG_JSONL      = RAW_DIR / "registration_log.jsonl"
EDITS_JSONL    = RAW_DIR / "newcomer_edits.jsonl"
TALKPAGE_JSONL = RAW_DIR / "talkpage_messages.jsonl"

# ── Clear stale data in test mode ─────────────────────────────────────────────
if TEST_MODE:
    for path in (LOG_JSONL, EDITS_JSONL, TALKPAGE_JSONL):
        if path.exists():
            path.unlink()
            print(f"Cleared {path.name}")

# ── Module-level session ──────────────────────────────────────────────────────
# A Session reuses the underlying TCP connection across calls (connection
# pooling) and lets us set the User-Agent header once for all requests.
# Wikimedia policy requires a descriptive User-Agent with contact info.
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "BDACA-II-course/1.0 (educational; adashiyj@gmail.com)"
})

print(f"requests {requests.__version__}")
print(f"pandas   {pd.__version__}")
print(f"Session ready — User-Agent: {SESSION.headers['User-Agent']}")
if TEST_MODE:
    print(f"TEST MODE — window: {LOG_START} → {LOG_END}, cap: {MAX_LOG_ACCOUNTS}")

requests 2.32.5
pandas   2.3.3
Session ready — User-Agent: BDACA-II-course/1.0 (educational; adashiyj@gmail.com)


### 1.1 Define the API wrapper

In [3]:
def api_get(
    params: dict,
    api: str = API,
    max_retries: int = MAX_RETRIES,
    maxlag: int = MAXLAG,
) -> dict:
    """GET the MediaWiki JSON API via the module-level session.

    All API calls in this project go through this wrapper. It handles
    cooperative rate limiting, transient server errors, and API-level
    error responses so that callers never need to implement retry logic
    themselves.

    Parameters
    ----------
    params      : MediaWiki API query parameters (without maxlag).
                  A shallow copy is made internally — the caller's dict
                  is never mutated.
    api         : Base URL of the MediaWiki API endpoint. Defaults to
                  the English Wikipedia API defined at module level.
    max_retries : Maximum number of retry attempts before raising a
                  RuntimeError. Defaults to MAX_RETRIES.
    maxlag      : Cooperative DB-lag threshold in seconds. The API returns
                  HTTP 503 when the replica DB lags beyond this value,
                  which triggers automatic back-off. Defaults to MAXLAG.

    Returns
    -------
    dict
        Parsed JSON response from the API.

    Raises
    ------
    RuntimeError
        If the API returns an error body (even with HTTP 200), or if all
        retry attempts are exhausted.
    """
    # Copy params so we never mutate the caller's dict.
    call_params = {**params, "maxlag": str(maxlag)}

    for attempt in range(1, max_retries + 1):
        try:
            response = SESSION.get(
                api,
                params=call_params,
                timeout=REQUEST_TIMEOUT,
            )

            # Retry on transient server-side errors before raising.
            if response.status_code in (429, 503, 504):
                print(
                    f"  HTTP {response.status_code} "
                    f"(attempt {attempt}/{max_retries}), "
                    f"waiting {RETRY_DELAY}s ..."
                )
                time.sleep(RETRY_DELAY)
                continue

            # Raise immediately for non-retryable HTTP errors (e.g. 404, 400).
            response.raise_for_status()
            result = response.json()

            # The MediaWiki API can return error details inside an HTTP 200
            # response — raise explicitly so callers see a meaningful message.
            if "error" in result:
                err = result["error"]
                raise RuntimeError(
                    f"MediaWiki API error [{err.get('code', '?')}]: "
                    f"{err.get('info', err)}"
                )
            if "errors" in result:
                messages = "; ".join(
                    e.get("text", str(e)) for e in result["errors"]
                )
                raise RuntimeError(f"MediaWiki API errors: {messages}")

            return result

        except requests.exceptions.Timeout:
            # Timeout is retryable — the server may have been temporarily busy.
            print(
                f"  Timeout on attempt {attempt}/{max_retries}, "
                f"waiting {RETRY_DELAY}s ..."
            )
            time.sleep(RETRY_DELAY)

        except requests.exceptions.ConnectionError as exc:
            # Connection errors (e.g. dropped socket) are also retryable.
            print(
                f"  Connection error (attempt {attempt}/{max_retries}): "
                f"{exc!r}, waiting {RETRY_DELAY}s ..."
            )
            time.sleep(RETRY_DELAY)

    raise RuntimeError(
        f"api_get() failed after {max_retries} retries for params: {params}"
    )

### 1.2 Define JSON helpers

In [4]:
def append_jsonl(record: dict, path: Path) -> None:
    """Append one JSON record as a single line to a JSONL file.
 
    Opening in append mode means a crash mid-collection loses nothing already
    written. Safe to call repeatedly; creates the file if absent.
 
    Parameters
    ----------
    record : JSON-serialisable dict to write.
    path   : Target JSONL file path.
    """
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
 
 
def read_jsonl(path: Path) -> list[dict]:
    """Read all records from a JSONL file.
 
    Parameters
    ----------
    path : JSONL file path.
 
    Returns
    -------
    List of dicts, one per non-empty line. Returns [] if file does not exist.
    """
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
 
 
def already_collected_keys(path: Path, key: str) -> set:
    """Return the set of values already written for a given key field.
 
    Used to skip re-fetching items that survived a previous interrupted run.
 
    Parameters
    ----------
    path : JSONL file path.
    key  : Dict key whose values form the "already done" set
           (e.g., "username" or "user_id").
 
    Returns
    -------
    Set of key values already present in the file.
    """
    return {rec[key] for rec in read_jsonl(path) if key in rec}

### 1.3 Collect registration log data

In [5]:
def fetch_registration_log_page(params: dict) -> tuple[list, dict | None]:
    """Fetch one page of the new-user registration log.
 
    Parameters
    ----------
    params : MediaWiki API parameters, including any continuation token.
 
    Returns
    -------
    Tuple of (list of log entries, continuation dict or None if last page).
    """
    data = api_get(params)
    entries = data.get("query", {}).get("logevents", [])
    continuation = data.get("continue")  # None on the final page
    print(f"  Fetched page, got {len(entries)} entries, continuation={continuation is not None}")
    return entries, continuation
 
 
def collect_registration_log(
    start: str = LOG_START,
    end: str = LOG_END,
    max_accounts: int = MAX_LOG_ACCOUNTS,
    output_path: Path = LOG_JSONL,
) -> list[dict]:
    """Fetch newcomer accounts from Special:Log/newusers within a date window.
 
    Fetches the log sequentially, saves each entry immediately (crash-safe),
    and stops after max_accounts entries to keep collection manageable.
    Temporary accounts (username contains '~') are skipped during collection
    because they cannot be tracked across sessions.
 
    Parameters
    ----------
    start        : ISO 8601 start timestamp (inclusive).
    end          : ISO 8601 end timestamp (inclusive).
    max_accounts : Hard cap on entries fetched (3× the analysis target).
    output_path  : JSONL file to append results to.
 
    Returns
    -------
    List of all collected log entry dicts (including previously saved ones).
    """
    already_done = already_collected_keys(output_path, key="user_id")
    print(
        f"Registration log: {len(already_done)} accounts already collected. "
        f"Cap: {max_accounts}."
    )
 
    params = {
        "action":      "query",
        "list":        "logevents",
        "letype":      "newusers",
        "lestart":     end,    # MediaWiki lists newest first; lestart = upper bound
        "leend":       start,  # leend = lower bound
        "lelimit":     "500",  # maximum per call for authenticated users
        "leprop":      "ids|title|type|user|timestamp",
        "format":      "json",
        "formatversion": "2",
    }
 
    collected = len(already_done)
 
    while collected < max_accounts:
        entries, continuation = fetch_registration_log_page(params)
 
        if not entries:
            print("  Registration log exhausted.")
            break
 
        skipped_temp = 0
        for entry in entries:
            if collected >= max_accounts:
                break

            username = entry.get("user", "")

            if "~" in username:
                skipped_temp += 1
                continue

            user_id = entry.get("logid")

            if user_id in already_done:
                continue

            record = {
                "user_id":   user_id,
                "username":  username,
                "timestamp": entry.get("timestamp", ""),
            }
            append_jsonl(record, output_path)
            already_done.add(user_id)
            collected += 1

        print(f"  Skipped {skipped_temp}/500 temporary accounts this page")
 
        if collected % 1000 == 0:
            print(f"  Progress: {collected} accounts collected.")
 
        if continuation is None:
            print("  No further continuation — log fully fetched.")
            break
 
        # Merge continuation token into params for the next page.
        params.update(continuation)
        time.sleep(INTER_PAGE_DELAY)  # polite delay between pages
 
    print(f"Registration log collection complete: {collected} accounts.")
    return read_jsonl(output_path)
 

In [6]:
# Quick-look: registration log
log_records = collect_registration_log()
print(f"Collected {len(log_records):,} accounts")
print("First record:", log_records[0] if log_records else 'n/a')

Registration log: 28491 accounts already collected. Cap: 30000.
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 0/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 1/500 temporary accounts this page
  Fetched page, got 500 entries, continuation=True
  Skipped 1/500 tempo

### 1.4 Collect edit histories

In [9]:
def fetch_user_contributions(
    username: str,
    start: str,
    end: str,
    limit: int = 500,
) -> list[dict]:
    """Fetch all edits made by a user within a given time window.
 
    Follows continuation tokens so the full edit history is retrieved even
    when it spans multiple API pages.
 
    Parameters
    ----------
    username : Wikipedia username.
    start    : ISO 8601 lower bound (oldest edit to include).
    end      : ISO 8601 upper bound (newest edit to include).
    limit    : Maximum edits per API call (500 = API maximum).
 
    Returns
    -------
    List of contribution dicts, each with keys: revid, timestamp, title,
    ns (namespace), minor, sizediff.
    """
    params = {
        "action":    "query",
        "list":      "usercontribs",
        "ucuser":    username,
        "ucstart":   end,      # newest first (descending order)
        "ucend":     start,
        "uclimit":   str(limit),
        "ucprop":    "ids|timestamp|title|sizediff|flags",
        "format":    "json",
        "formatversion": "2",
    }
 
    all_contributions = []
 
    while True:
        data = api_get(params)
        contributions = data.get("query", {}).get("usercontribs", [])
        all_contributions.extend(contributions)
 
        if "continue" not in data:
            break
        params.update(data["continue"])
 
    return all_contributions
 
 
def compute_session_edits(contributions: list[dict], session_minutes: int = SESSION_WINDOW_MIN) -> int:
    """Count edits made within the first edit session.
 
    The first edit session is defined as the SESSION_WINDOW_MIN-minute window
    starting from the timestamp of the newcomer's very first edit, following
    Halfaker et al. (2013) and Zheng et al. (2019).
 
    Parameters
    ----------
    contributions   : List of contribution dicts (oldest-first order assumed
                      after sorting by timestamp).
    session_minutes : Duration of the first session window in minutes.
 
    Returns
    -------
    Number of edits made within the session window. Returns 0 if no edits.
    """
    if not contributions:
        return 0
 
    # Sort ascending so the first element is the earliest edit.
    sorted_contribs = sorted(contributions, key=lambda c: c["timestamp"])
    first_edit_time = datetime.fromisoformat(
        sorted_contribs[0]["timestamp"].replace("Z", "+00:00")
    )
 
    session_count = 0
    for contrib in sorted_contribs:
        edit_time = datetime.fromisoformat(
            contrib["timestamp"].replace("Z", "+00:00")
        )
        elapsed_minutes = (edit_time - first_edit_time).total_seconds() / 60
        if elapsed_minutes <= session_minutes:
            session_count += 1
        else:
            break  # sorted, so no later edits can fall inside the window
 
    return session_count
 
 
def collect_edit_histories(
    log_records: list[dict],
    output_path: Path = EDITS_JSONL,
) -> list[dict]:
    """Fetch editing histories for all newcomers in the registration log.
 
    For each newcomer:
      - Fetches all edits in the collection window.
      - Skips accounts with zero edits (not active newcomers in scope).
      - Computes session_edits (control variable).
      - Records first_edit_timestamp (needed for retention calculation).
 
    Parameters
    ----------
    log_records : List of registration log records (from collect_registration_log).
    output_path : JSONL file to append results to.
 
    Returns
    -------
    List of all collected edit history records.
    """
    already_done = already_collected_keys(output_path, key="user_id")
    print(
        f"Edit histories: {len(already_done)} already collected. "
        f"Total to process: {len(log_records)}."
    )
 
    for i, newcomer in enumerate(log_records):
        user_id  = newcomer["user_id"]
        username = newcomer["username"]
 
        if user_id in already_done:
            continue

        # Skip records with empty username — would cause an API error.
        if not username:
            continue
 
        contributions = fetch_user_contributions(
            username,
            start=LOG_START,
            end=LOG_END,
        )
 
        # Skip accounts with zero edits — not active in the collection window.
        if not contributions:
            # Still record the account so we know it was processed.
            append_jsonl(
                {"user_id": user_id, "username": username, "total_edits": 0},
                output_path,
            )
            already_done.add(user_id)
            continue
 
        session_edits = compute_session_edits(contributions)
 
        sorted_contribs = sorted(contributions, key=lambda c: c["timestamp"])
        first_edit_ts   = sorted_contribs[0]["timestamp"]
 
        record = {
            "user_id":            user_id,
            "username":           username,
            "total_edits":        len(contributions),
            "session_edits":      session_edits,
            "first_edit_timestamp": first_edit_ts,
        }
        append_jsonl(record, output_path)
        already_done.add(user_id)
 
        if i % 500 == 0:
            print(f"  Processed {i}/{len(log_records)} newcomers.")
 
        time.sleep(INTER_REQUEST_DELAY)  # polite delay per user
 
    print("Edit history collection complete.")
    return read_jsonl(output_path)

In [10]:
# Quick-look: edit histories
edit_records = collect_edit_histories(log_records)
active = [r for r in edit_records if r.get('total_edits', 0) > 0]
print(f"Total processed : {len(edit_records):,}")
print(f"Active (≥1 edit): {len(active):,}  ({100*len(active)/max(len(edit_records),1):.1f}%)")

Edit histories: 7307 already collected. Total to process: 30000.
  Processed 8500/30000 newcomers.
  Processed 9500/30000 newcomers.
  Timeout on attempt 1/5, waiting 2s ...
  Processed 11000/30000 newcomers.
  Processed 12500/30000 newcomers.
  Timeout on attempt 1/5, waiting 2s ...
  Timeout on attempt 1/5, waiting 2s ...
  Timeout on attempt 1/5, waiting 2s ...
  Processed 22500/30000 newcomers.
  Processed 24500/30000 newcomers.
  Processed 25500/30000 newcomers.
  Processed 27500/30000 newcomers.
Edit history collection complete.
Total processed : 29,996
Active (≥1 edit): 5,845  (19.5%)


### 1.5 Sampling

In [11]:
def sample_newcomers(
    edit_records: list[dict],
    target_n: int = TARGET_SAMPLE_SIZE,
    seed: int = RANDOM_SEED,
) -> list[dict]:
    """Stratified random sample of active newcomers, balanced by registration month.
 
    Active newcomers are those with total_edits > 0. Stratification by
    registration month prevents over-representation of any single month
    (following Zheng et al., 2019).
 
    Parameters
    ----------
    edit_records : List of edit history records.
    target_n     : Desired final sample size.
    seed         : Random seed for reproducibility.
 
    Returns
    -------
    List of sampled newcomer records (length ≤ target_n).
    """
    active = [r for r in edit_records if r.get("total_edits", 0) > 0]
 
    if len(active) < target_n:
        print(
            f"Warning: only {len(active)} active newcomers available; "
            f"returning all (target was {target_n})."
        )
        return active
 
    # Group by registration month for stratified sampling.
    by_month: dict[str, list] = {}
    for record in active:
        # Derive month from first_edit_timestamp (proxy for registration month).
        month = record.get("first_edit_timestamp", "")[:7]  # "YYYY-MM"
        by_month.setdefault(month, []).append(record)
 
    n_months        = len(by_month)
    per_month_quota = target_n // n_months
 
    rng     = random.Random(seed)
    sampled = []
 
    for month, group in sorted(by_month.items()):
        pool = list(group)
        rng.shuffle(pool)
        sampled.extend(pool[:per_month_quota])
 
    # Top up to exactly target_n if rounding left us short.
    remaining = [r for r in active if r not in sampled]
    rng.shuffle(remaining)
    sampled.extend(remaining[: target_n - len(sampled)])
 
    print(f"Sample: {len(sampled)} newcomers drawn from {n_months} months.")
    return sampled[:target_n]
 

In [12]:
# Quick-look: stratified sample
sample = sample_newcomers(edit_records)
months = {r.get('first_edit_timestamp','')[:7] for r in sample}
print(f"Sample size : {len(sample):,}")
print(f"Months span : {sorted(months)}")

Sample size : 5,845
Months span : ['2025-03', '2025-04', '2025-05']


### 1.6 Collect talk page messages

In [13]:
def fetch_talkpage_revisions(username: str, end_timestamp: str) -> list[dict]:
    """Fetch all content-adding revisions of a user's talk page.
 
    Only revisions that added content (positive sizediff) within the first
    two months after the newcomer's first edit are returned. Each revision
    records the sender's username for later bot-role matching.
 
    Parameters
    ----------
    username      : Wikipedia username (talk page = "User talk:<username>").
    end_timestamp : ISO 8601 upper bound (two months after first edit).
 
    Returns
    -------
    List of revision dicts with keys: revid, timestamp, sender, sizediff.
    """
    talkpage_title = f"User talk:{username}"
 
    params = {
        "action":    "query",
        "titles":    talkpage_title,
        "prop":      "revisions",
        "rvprop":    "ids|timestamp|user|size|flags",
        "rvlimit":   "max",
        "rvend":     end_timestamp,   # stop at end of two-month window
        "rvdir":     "newer",         # oldest first for easier windowing
        "format":    "json",
        "formatversion": "2",
    }
 
    all_revisions = []
 
    while True:
        data      = api_get(params)
        pages     = data.get("query", {}).get("pages", [])
 
        if not pages or pages[0].get("missing"):
            break  # talk page does not exist — newcomer received no messages
 
        revisions = pages[0].get("revisions", [])
        all_revisions.extend(revisions)
 
        if "continue" not in data:
            break
        params.update(data["continue"])
 
    # Retain only revisions that added content (positive sizediff indicates
    # new text was written, not just formatting or deletion).
    content_adding = [
        {
            "revid":     rev.get("revid"),
            "timestamp": rev.get("timestamp", ""),
            "sender":    rev.get("user", ""),
            "sizediff":  rev.get("size", 0),
        }
        for rev in all_revisions
        if rev.get("size", 0) > 0 and not rev.get("userhidden")
    ]
    return content_adding
 
 
def collect_talkpage_messages(
    edit_records: list[dict],
    output_path: Path = TALKPAGE_JSONL,
) -> list[dict]:
    """Fetch talk page messages received by each newcomer in the first two months.
 
    For each newcomer with at least one edit:
      - Computes the two-month window from first_edit_timestamp.
      - Fetches all content-adding talk page revisions in that window.
      - Records the sender of each message for bot-role matching.
 
    Parameters
    ----------
    edit_records : List of edit history records (from collect_edit_histories).
    output_path  : JSONL file to append results to.
 
    Returns
    -------
    List of all collected talk page message records.
    """
    # Only process newcomers who made at least one edit.
    active_newcomers = [r for r in edit_records if r.get("total_edits", 0) > 0]
 
    already_done = already_collected_keys(output_path, key="user_id")
    print(
        f"Talk pages: {len(already_done)} already collected. "
        f"Active newcomers to process: {len(active_newcomers)}."
    )
 
    for i, newcomer in enumerate(active_newcomers):
        user_id  = newcomer["user_id"]
        username = newcomer["username"]
 
        if user_id in already_done:
            continue
 
        # Compute the end of the two-month retention window.
        first_edit = datetime.fromisoformat(
            newcomer["first_edit_timestamp"].replace("Z", "+00:00")
        )
        # Add RETENTION_DAYS using timedelta for correctness.
        window_end = first_edit + timedelta(days=RETENTION_DAYS)
        window_end_str = window_end.strftime("%Y-%m-%dT%H:%M:%SZ")
 
        messages = fetch_talkpage_revisions(username, end_timestamp=window_end_str)
 
        record = {
            "user_id":       user_id,
            "username":      username,
            "message_count": len(messages),
            "messages":      messages,  # list of {revid, timestamp, sender, sizediff}
        }
        append_jsonl(record, output_path)
        already_done.add(user_id)
 
        if i % 500 == 0:
            print(f"  Processed {i}/{len(active_newcomers)} talk pages.")
 
        time.sleep(INTER_REQUEST_DELAY)
 
    print("Talk page collection complete.")
    return read_jsonl(output_path)

In [14]:
# Quick-look: talk page messages
talkpage_records = collect_talkpage_messages(sample)
with_msgs = [r for r in talkpage_records if r.get('message_count', 0) > 0]
print(f"Newcomers processed      : {len(talkpage_records):,}")
print(f"Received ≥1 message      : {len(with_msgs):,}  ({100*len(with_msgs)/max(len(talkpage_records),1):.1f}%)")
avg = sum(r['message_count'] for r in talkpage_records) / max(len(talkpage_records), 1)
print(f"Avg messages per newcomer: {avg:.2f}")

Talk pages: 0 already collected. Active newcomers to process: 5845.
  Processed 0/5845 talk pages.
  Processed 500/5845 talk pages.
  Processed 1000/5845 talk pages.
  Processed 1500/5845 talk pages.
  Processed 2000/5845 talk pages.
  Processed 2500/5845 talk pages.
  Processed 3000/5845 talk pages.
  Processed 3500/5845 talk pages.
  Processed 4000/5845 talk pages.
  Processed 4500/5845 talk pages.
  Processed 5000/5845 talk pages.
  Processed 5500/5845 talk pages.
Talk page collection complete.
Newcomers processed      : 5,845
Received ≥1 message      : 2,571  (44.0%)
Avg messages per newcomer: 82.01


### 1.7 Wikipedia data collection pipeline

In [15]:
def run_collection_pipeline() -> None:
    """Execute the full data collection pipeline end-to-end.
 
    Steps
    -----
    1. Fetch the new-user registration log (up to MAX_LOG_ACCOUNTS entries).
    2. Fetch editing histories for each registered account; skip zero-edit accounts.
    3. Draw a stratified random sample of TARGET_SAMPLE_SIZE active newcomers.
    4. Fetch talk page messages for all sampled newcomers.
    5. Save the final sample index to disk for use in downstream notebooks.
 
    All intermediate outputs are saved to JSONL files immediately, so the
    pipeline can be safely interrupted and resumed from any step.
    """
    print("=" * 60)
    print("Step 1: Collecting registration log")
    print("=" * 60)
    log_records = collect_registration_log()
 
    print("\n" + "=" * 60)
    print("Step 2: Collecting edit histories")
    print("=" * 60)
    edit_records = collect_edit_histories(log_records)
 
    print("\n" + "=" * 60)
    print("Step 3: Drawing stratified sample")
    print("=" * 60)
    sample = sample_newcomers(edit_records)
 
    # Persist the sample index so downstream notebooks use exactly these users.
    sample_path = PROCESSED_DIR / "sample_index.json"
    with sample_path.open("w", encoding="utf-8") as f:
        json.dump(sample, f, ensure_ascii=False, indent=2)
    print(f"Sample index saved to {sample_path}.")
 
    print("\n" + "=" * 60)
    print("Step 4: Collecting talk page messages")
    print("=" * 60)
    collect_talkpage_messages(sample)
 
    print("\n" + "=" * 60)
    print("Collection pipeline complete.")
    print(f"  Registration log : {LOG_JSONL}")
    print(f"  Edit histories   : {EDITS_JSONL}")
    print(f"  Talk page msgs   : {TALKPAGE_JSONL}")
    print(f"  Sample index     : {sample_path}")
    print("=" * 60)

In [16]:
# Run the full pipeline 
start = time.time()
run_collection_pipeline()
print(f"Total time: {time.time() - start:.1f}s")

Step 1: Collecting registration log
Registration log: 30000 accounts already collected. Cap: 30000.
Registration log collection complete: 30000 accounts.

Step 2: Collecting edit histories
Edit histories: 29996 already collected. Total to process: 30000.
Edit history collection complete.

Step 3: Drawing stratified sample
Sample index saved to Final_project/data/processed/sample_index.json.

Step 4: Collecting talk page messages
Talk pages: 5845 already collected. Active newcomers to process: 5845.
Talk page collection complete.

Collection pipeline complete.
  Registration log : Final_project/data/raw/registration_log.jsonl
  Edit histories   : Final_project/data/raw/newcomer_edits.jsonl
  Talk page msgs   : Final_project/data/raw/talkpage_messages.jsonl
  Sample index     : Final_project/data/processed/sample_index.json
Total time: 1.7s
